# **Multimodal Transformers**

The transformer's attention mechanism is **modality-agnostic**: it operates on a sequence of vectors (tokens) and doesn't care whether those vectors came from words, image patches, or audio frames. This is why the transformer became the universal backbone for multimodal models — if every modality can be turned into a sequence of tokens, a single attention stack can mix them.

## **1. Tokenizing different modalities**

The first step is converting each modality into tokens that can share a sequence:

- **Text** — subword tokens (BPE/WordPiece) + an embedding lookup.
- **Images** — split into fixed patches (e.g. 16×16), linearly project each patch into a vector (the ViT recipe). Each patch is a "visual token".
- **Audio** — patches of a spectrogram, or frames from a speech encoder.

Tokens from different modalities are typically tagged with **modality-type embeddings** and their own **positional encodings**, then either concatenated into one sequence or kept in separate streams that interact via attention.

## **2. Cross-attention vs co-attention vs single-stream**

Three common ways to let modalities interact:

- **Single-stream (merged) attention** — concatenate all tokens into one sequence and run standard self-attention (e.g. **ViLT**, **UNITER**). Every token can attend to every other token regardless of modality. Simple and powerful, but cost grows with the *total* sequence length.
- **Cross-attention** — keep modality A as the **query** and let it attend into modality B as **keys/values** (queries from text, keys/values from image). Information flows in one direction per layer; used in generative VLMs and Flamingo.
- **Co-attention (two-stream)** — two parallel transformer streams that exchange information through *symmetric* cross-attention layers (each modality attends to the other). This is the **ViLBERT** design.

## **3. Handling the cost: Perceiver and learned queries**

Merged self-attention is $O(L^2)$ in the combined sequence length $L$, which explodes for high-resolution images or long audio. Two influential remedies:

- **Perceiver / Perceiver IO** — a small set of learned **latent** vectors cross-attend to the (large) input array, compressing it into a fixed-size latent bottleneck; subsequent self-attention runs only over the latents. Cost becomes *linear* in input size and is modality-independent, so the same architecture handles images, audio, point clouds, or text.
- **Q-Former (BLIP-2)** — a similar idea specialized for vision→language: a fixed number of learned query tokens distill image features into a compact set the LLM can consume.

## **4. Toward unified models**

Recent work pushes a *single* transformer to handle many modalities and tasks by serializing everything into one token vocabulary/sequence:

- **Unified-IO**, **OFA**, **Gato** — cast detection, captioning, VQA, segmentation (and even control) as sequence-to-sequence over a shared token space.
- **ImageBind** — binds *six* modalities (image, text, audio, depth, thermal, IMU) into one embedding space using image-paired data, enabling emergent cross-modal retrieval.

The trend: fewer modality-specific components, more shared parameters, with modality differences pushed into the **tokenizer/embedding** layer rather than the core network.

## **5. Sketch: building a mixed-modality token sequence**

```python
import torch
import torch.nn as nn

class MixedSequence(nn.Module):
    def __init__(self, dim, n_text_vocab, patch_dim):
        super().__init__()
        self.text_embed = nn.Embedding(n_text_vocab, dim)
        self.patch_proj = nn.Linear(patch_dim, dim)   # ViT-style patch projection
        self.modality_type = nn.Embedding(2, dim)      # 0 = text, 1 = image

    def forward(self, text_ids, image_patches):
        t = self.text_embed(text_ids) + self.modality_type.weight[0]
        v = self.patch_proj(image_patches) + self.modality_type.weight[1]
        # concatenate into ONE sequence -> feed to a standard TransformerEncoder
        return torch.cat([t, v], dim=1)               # (B, L_text + L_patches, dim)
```

**References**
- Lu et al., *ViLBERT*, 2019.
- Kim et al., *ViLT: Vision-and-Language Transformer Without Convolution or Region Supervision*, 2021.
- Jaegle et al., *Perceiver* / *Perceiver IO*, 2021.
- Girdhar et al., *ImageBind*, 2023.